In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random as rand
import math as math
from collections import Counter
import statistics as statistics

In [ ]:
def expected_score(playerA, playerB):
    rA = playerA.init_rating + rand.uniform(-1,1) * playerA.volatility + 35
    rB = playerB.init_rating + rand.uniform(-1,1) * playerB.volatility
    return 1 / (1 + 10 ** ((rB - rA) / 400))

def draw_chance(playerA, playerB):
    rA = playerA.init_rating + rand.uniform(-1,1) * playerA.volatility + 35
    rB = playerB.init_rating + rand.uniform(-1,1) * playerB.volatility
    return max(0.35, 0.5 - abs(rA-rB)/800) 

def expected_score_rating(ratingA, ratingB):
    return float(1 / (1 + 10 ** ((ratingB - (ratingA + 35)) / 400)))

def countList(list, value, start, end):
    vals = list.copy()
    num = 0
    for i in range(start, end):
        if vals[i] == value:
            num += 1
    return num


In [ ]:
playersList = []
winnersList = []
tournamentWinnersList = []
scoresList = []
tiebreaksList = []
pairingsList = []
ratingsList = []

In [ ]:
class Player: 
    def __init__(self, name, rating, volatility = 0):

        self.init_rating = rating
        self.name = name
        self.rating = rating
        self.volatility = volatility

        self.gameResults = [] # results of individual games
        self.tournamentResults = [] # results (place) of individual tournaments
        self.tournamentScores = [] # scores of individiual tournaments
        self.winList = [] # opponents won against
        self.lossList = [] # opponents lost against
        self.drawList = [] # opponents drew against
        self.opponentsList = [] # opponents
        self.colorsList = [] # colors vs opponents
        self.newRatingsList = [] # new ratings after a tournament
        self.tprList = [] # TPR for each tournament

        self.tiebreak = 0 # tiebreak score
        self.wins = 0 # num wins
        self.losses = 0 # num losses
        self.draws = 0 # num draws
        self.score = 0.0 # score
        self.whites = 0 # num white games
        self.blacks = 0 # num black games
        self.tpr = 0 # tournament performance rating

        playersList.append(self)
        self.index = len(playersList) - 1  
        # DO NOT SWAP ORDER OF THESE LINES - OTHERWISE EVERYTHING BREAKS



        ratingsList.append(self.init_rating)
        tournamentWinnersList.append(0)

    def resetAll(self):
        self.wins = 0
        self.losses = 0
        self.draws = 0
        self.tiebreak = 0
        self.score = 0
        self.whites = 0
        self.blacks = 0
        self.tpr = 0
        self.rating = self.init_rating
        
        self.gameResults.clear()
        self.winList.clear()
        self.lossList.clear()
        self.drawList.clear()
        self.opponentsList.clear()
        self.colorsList.clear()
        
    def setScore(self):
        self.score = sum(self.gameResults)
        self.tournamentScores.append(self.score)

    def getRecordString(self):
       return f"{self.wins} - {self.losses} - {self.draws}"
    
    def getScoreString(self):
        return f"{self.score}/{float(self.wins + self.losses + self.draws)}"
    
    def calculateTiebreak(self):
        wins = self.winList
        draws = self.drawList
        if len(wins) > 0:
            for opp in wins:
                self.tiebreak += opp.score
        if len(draws) > 0:
            for opp in draws:
                self.tiebreak += (opp.score * 0.5)

    def analyzeRankingsBar(self):
        n = len(playersList)
        vals = np.array(self.tournamentResults)
        mean = vals.mean()
        stdev = vals.std()

        frequency = Counter(self.tournamentResults)
        places = np.arange(1, n+1)
        results = []
        for i in range(n):
            results.append(frequency[i+1])
        percents = np.array(results)
        percents = (percents / percents.sum())

        colorMap = plt.colormaps['RdYlGn_r']
        norm = plt.Normalize(min(places), max(places))
        barColors = colorMap(norm(places))
        plt.xticks(places)
        plt.bar(places, percents, color = barColors)
        plt.title(f"{self.name} Tournament Results ({round(self.init_rating)})")
        plt.text(0.95, 0.95, rf"$\mu = {mean:.2f}$" "\n" rf"$\sigma = {stdev:.2f}$", ha="right", va="top",transform=plt.gca().transAxes)
        plt.show()

    def analyzePerformanceLine(self):
        binSize = 10

        num_x_ticks = round((math.log(sum(tournamentWinnersList)) / math.log(binSize))) # logarithmic scale for x vals
        num_wins = np.zeros(num_x_ticks)
        for i in range (len(num_wins)):
            num_wins[i] = (binSize ** (i+1))
        num_wins = num_wins.astype(int)

        results = self.tournamentResults.copy()
        tourneysWon = [None] * num_x_ticks
        tourneysWon[0] = results[:10].count(1)
        for i in range (num_x_ticks - 1):
            tourneysWon[i + 1] = tourneysWon[i] + countList(results, 1, binSize ** (i+1), (binSize ** (i+2)))
            
        percents = np.array(tourneysWon) / num_wins

        plt.plot(num_wins, percents, marker = "o")
        plt.xticks(num_wins)
        plt.xscale("log")
        plt.xlabel("Number of Tournament Simulations")
        plt.ylabel("Proportion of Tournaments Won")
        plt.ylim(0, 1)
        plt.title(f"{self.name} Convergence of Tournament Win Rate")
        plt.show()

    def analyzeScoresBar(self):
        n = len(playersList)
        vals = np.array(self.tournamentScores)
        mean = vals.mean()
        stdev = vals.std()
        se = float(stdev / (len(vals) ** 0.5))
                
        print(f"Observed: {mean:.4f}")
        print(f"95% CI: [{mean - 1.96*se:.4f}, {mean + 1.96*se:.4f}]")

        frequency = Counter(self.tournamentScores)
        scores = (np.arange(1, 2 * n))/2
        ticks = np.arange(n)
        results = []
        for i in range(2 * n - 1):
            results.append(frequency[i/2])

        percents = np.array(results)
        percents = (percents / percents.sum())

        colorMap = plt.colormaps['Blues']
        norm = plt.Normalize(min(scores), max(scores))
        barColors = colorMap(norm(scores))
        plt.xticks(ticks)
        plt.bar(scores, percents, color = barColors, width = 0.5)
        plt.title(f"{self.name} Tournament Scores ({round(self.init_rating)})")
        plt.text(0.95, 0.95, rf"$\mu = {mean:.2f}$" "\n" rf"$\sigma = {stdev:.2f}$", ha="right", va="top",transform=plt.gca().transAxes)
        plt.show()

    def calculateTPR(self):
        avgRating = statistics.mean(ratingsList)
        min = avgRating - 800
        max = avgRating + 800
        guess = avgRating
        for i in range(15):
            expectation = tournament_expected_score_rtg(guess, self.index)
            if math.fabs(expectation - self.score) < 0.01:
                break
            elif expectation > self.score:
                max = guess
            else:
                min = guess
            guess = (min + max)/2
        
        self.tprList.append(guess)
        # primitive implementation, very inefficient runtime
        return guess
    

    

            

In [ ]:

# P1 = Player("Wesley", 2754)
# P2 = Player("Gukesh", 2717)
# P3 = Player("Magnus", 2840)
# P4 = Player("Pragg", 2733)
# P5 = Player("Vincent", 2759)
# P6 = Player("Alireza", 2759)
# P7 = Player("Hikaru", 2800)
# P8 = Player("Fabi", 2790)

P1 = Player("Niko", 2634)
P2 = Player("Mukhiddin", 2586)
P3 = Player("Shak", 2717)
P4 = Player("Yakubbeov", 2689)
P5 = Player("Vokhidov", 2637)
P6 = Player("Abdusattorov", 2777)
P7 = Player("Arjun", 2761)
P8 = Player("Vidit", 2708)
P9 = Player("Hans", 2742)
P10 = Player("Nepo", 2733)

def resetAll():
    for i in range(len(playersList)):
        playersList[i].resetAll()
    

In [ ]:
n = len(playersList)
playerComboGrid = [[0 for i in range(n)] for j in range(n)]

for i in range(n):
    for j in range(n):
        expected = expected_score(playersList[i], playersList[j])
        p_draw = draw_chance(playersList[i], playersList[j])
        p_win = expected - (p_draw * 0.5)
        playerComboGrid[i][j] = (p_win, p_draw, expected)
        

In [ ]:
def updateRating(self):
        sI = self.index
        rating = self.rating
        
        for W in self.winList:
            oI = W.index
            # expectedPerf = expected_score(self.init_rating, playersList[W].init_rating)
            rating += 10 * (1 - playerComboGrid[sI][oI][2])
        for L in self.lossList:
            oI = L.index 
            # expectedPerf = expected_score(self.init_rating, playersList[L].init_rating)
            rating += 10 * (0 - playerComboGrid[oI][sI][2])
        for D in self.drawList:
            oI = D.index
            # expectedPerf = expected_score(self.init_rating, playersList[D].init_rating)
            rating += 10 * (0.5 - playerComboGrid[sI][oI][2])
            
        self.rating = rating

# sI is the self index, oI is the opponents' index (referring to index in playersList)

            
Player.updateRating = updateRating 


def tournament_expected_score(self, opponents):
    # make sure to enter playersList
    sI = self.index
    expected = 0
    for opp in opponents:
        if opp is not self:
            oI = opp.index
            expected += ( (playerComboGrid[sI][oI][2] + (1 - playerComboGrid[oI][sI][2]) ) / 2)
    print(f"Expected: {expected:.4f}")
    return (expected)

def tournament_expected_score_rtg(rating, index):
    expected = 0
    for player in playersList:
        if player.index != index:
            expected += ( (expected_score_rating(rating, player.init_rating) + (1 - expected_score_rating(player.init_rating, rating))) / 2 )
    return expected

Player.tournament_expected_score = tournament_expected_score

def score_diff(self, opponents, score):
    return self.tournament_expected_score(self.rating, opponents) - score

In [ ]:
def simulate_game(A,B):
    A.colorsList.append(0)
    B.colorsList.append(1)
    A.whites += 1
    B.blacks += 1
    A.opponentsList.append(B.index)
    B.opponentsList.append(A.index)
    p_a_win = playerComboGrid[A.index][B.index][0]
    p_draw = playerComboGrid[A.index][B.index][1]
    roll = rand.random()

    if roll < 1 - p_a_win - p_draw:
        # A.addLoss(B)
        A.losses += 1
        # A.gameResults.append("L")
        A.lossList.append(B)
        A.gameResults.append(0)

        # B.addWin(A)
        B.wins += 1
        # B.gameResults.append("W")
        B.winList.append(A)
        B.gameResults.append(1)

        # A.updateRating(ratingB, "L")
        # B.updateRating(ratingA, "W")
    elif roll < 1 - p_draw:
        # A.addWin(B)
        A.wins +=1
        # A.gameResults.append("W")
        A.winList.append(B)
        A.gameResults.append(1)

        # B.addLoss(A)
        B.losses += 1
        # B.gameResults.append("L")
        B.lossList.append(A)
        B.gameResults.append(0)
        
        # A.updateRating(ratingB, "W")
        # B.updateRating(ratingA, "L")
    else:
        # A.addDraw(B)
        A.draws += 1
        # A.gameResults.append("D")
        A.drawList.append(B)
        A.gameResults.append(0.5)

        # B.addDraw(A)
        B.draws +=1
        # B.gameResults.append("D")
        B.drawList.append(A)
        B.gameResults.append(0.5)


        # A.updateRating(ratingB, "D")
        # B.updateRating(ratingA, "D")
       
def simulate_round_robin(players, rounds):
    for k in range(rounds):
        for i in range(len(players)):
            for j in range(i+1, len(players)):
                roll = rand.random()
                if roll < 0.5:
                    simulate_game(players[i],players[j])
                else:
                    simulate_game(players[j],players[i])


def simulate_tiebreak_game(A,B):
    q = playerComboGrid[A.index][B.index][0]
    roll = rand.random()
    if roll >= q:
        A.losses += 1
        # A.gameResults.append("L")
        A.lossList.append(B)

        B.wins += 1
        # B.gameResults.append("W")
        B.winList.append(A)
        return B
    A.wins +=1
    # A.gameResults.append("W")
    A.winList.append(B)
    
    B.losses += 1
    # B.gameResults.append("L")
    B.lossList.append(A)



def simulate_knockout(players, rounds):
    winners = []
    current = players.copy()
    
    while len(current) > 1:
        result = False
        roll = math.floor(1 + rand.random()*(len(current)-1))

        while result == False:
            for i in range (rounds):
                simulate_game(current[0], current[roll])
                simulate_game(current[roll], current[0])
            current[0].setScore()
            current[roll].setScore()
            if current[0].score > 2:
                winners.append(current[0])
                result = True
            elif current[roll].score > 2:
                winners.append(current[roll])
                result = True
            current[0].resetAll()
            current[roll].resetAll()

        current.pop(roll)
        current.pop(0)
        if len(current) == 0 and len(winners) > 1:
            current = winners.copy()
            winners.clear()

    tournamentWinnersList[winners[0].index] += 1


def simulate_swiss(players):
    round = 1
    participants = players.copy()
    participants.sort(key = lambda player: player.rating, reverse=True)
    names = []
    for p in participants:
        names.append(p.name)
    print(participants)
    print(names)
    

    n = len(participants)

    for i in range (int(n/2)):
        pairingsList.append((participants[i], participants[n-1-i]))

    for A,B in pairingsList:
        simulate_game(A,B)

In [ ]:
simulate_swiss(playersList)

In [ ]:
def orderPlayers(players):
    names = []
    
    for player in players:
        scoresList.append(player.score)
        tiebreaksList.append(player.tiebreak)
        names.append(player.name)
    
    npScores = np.array(scoresList)
    npTiebreaks = np.array(tiebreaksList)
    npPlayers = np.array(players)
    npNames = np.array(names)

    indices = np.argsort(npTiebreaks)
    pre_sorted_scores = npScores[indices]
    final_indices = indices[np.argsort(pre_sorted_scores)]

    npScores = npScores[final_indices]
    npTiebreaks = npTiebreaks[final_indices] 
    npPlayers = npPlayers[final_indices]
    npNames = npNames[final_indices]

    for i in range (len(npPlayers)):
        npPlayers[i].tournamentResults.append(10-i)
    winnersList.append(npNames[-1])
    tournamentWinnersList[npPlayers[-1].index] += 1
    
    scoresList.clear()
    tiebreaksList.clear()



# def orderPlayers(players):
#     names = []
#     participants = players
#     for p in participants:
#         noise = rand.uniform(0, 0.0001)
#         p.rank_score = p.score + p.tiebreak * 0.001 + noise

#         # scoresList.append(player.score)
#         # tiebreaksList.append(player.tiebreak)
#         # names.append(player.name)
    
#     # npScores = np.array(scoresList)
#     # npTiebreaks = np.array(tiebreaksList)
#     # npPlayers = np.array(players)
#     # npNames = np.array(names)

#     # indices = np.argsort(npTiebreaks)
#     # pre_sorted_scores = npScores[indices]
#     # final_indices = indices[np.argsort(pre_sorted_scores)]

#     # npScores = npScores[final_indices]
#     # npTiebreaks = npTiebreaks[final_indices] 
#     # npPlayers = npPlayers[final_indices]
#     # npNames = npNames[final_indices]

#     participants.sort(key=lambda p: p.rank_score)
    
#     # for p in participants:
#     #     print (p.tiebreak)
#     #     print (p.rank_score)


#     for i in range (len(participants)):
#         participants[i].tournamentResults.append(10-i)
#     # winnersList.append(npNames[-1])
#     tournamentWinnersList[participants[-1].index] += 1
    
#     scoresList.clear()
#     tiebreaksList.clear()



In [ ]:
def simulateTournaments(count, format, players):
    if format == "RR": # round robin
        for i in range (count):
            resetAll()
            simulate_round_robin(players, 1)

            for i in range(len(players)):
                players[i].setScore()
                # players[i].calculateTPR()
                players[i].updateRating()

            for i in range(len(players)):
                players[i].calculateTiebreak()
            orderPlayers(players)


    elif format == "KO": #knock out
        for i in range (count):
            resetAll()
            simulate_knockout(players, 2)

            # for i in range(len(players)):
            #     players[i].updateRating()
            #     playersList[i].calculateTiebreak()
                

In [ ]:
simulateTournaments(100000, "RR", playersList)

for i in range(len(playersList)):
    print(playersList[i].name)
    # print(playersList[i].getRecordString())
    print(playersList[i].getScoreString()) 
    # print(playersList[i].rating)
    # print(playersList[i].tiebreak)

print(tournamentWinnersList)

In [ ]:
playerLabels = []
for i in range(len(playersList)):
    playerLabels.append(playersList[i].name)
print(playerLabels)
print(tournamentWinnersList)
plt.pie(tournamentWinnersList, labels=playerLabels, autopct='%1.1f%%')
plt.title("UzChess Cup 2026")
plt.show()

In [ ]:
def analyzeRankingsBarAll(players):
    participants = players.copy()
    participants.sort(key = lambda player: player.init_rating, reverse=True)
    n = len(participants)
    
    for p in participants:
        vals = np.array(p.tournamentResults)
        mean = vals.mean()
        stdev = vals.std()

        frequency = Counter(p.tournamentResults)
        places = np.arange(1, n+1)
        results = []
        for i in range(n):
            results.append(frequency[i+1])
        percents = np.array(results)
        percents = (percents / percents.sum())

        colorMap = plt.colormaps['RdYlGn_r']
        norm = plt.Normalize(min(places), max(places))
        barColors = colorMap(norm(places))
        plt.xticks(places)
        plt.bar(places, percents, color = barColors)
        plt.title(f"{p.name} Tournament Results ({round(p.init_rating)})")
        plt.text(0.95, 0.95, rf"$\mu = {mean:.2f}$" "\n" rf"$\sigma = {stdev:.2f}$", ha="right", va="top",transform=plt.gca().transAxes)
        plt.show()
        
def analyzePerformancesLineAll(players):
    binSize = 10
    participants = players.copy()
    participants.sort(key = lambda player: player.init_rating, reverse=True)

    num_x_ticks = round((math.log(sum(tournamentWinnersList)) / math.log(binSize))) # logarithmic scale for x vals
    num_wins = np.zeros(num_x_ticks) 
    for i in range (len(num_wins)):
        num_wins[i] = (binSize ** (i+1))
    num_wins = num_wins.astype(int)

    for p in participants:
        results = p.tournamentResults.copy()
        tourneysWon = [None] * num_x_ticks
        tourneysWon[0] = results[:10].count(1)
        for i in range (num_x_ticks - 1):
            tourneysWon[i + 1] = tourneysWon[i] + countList(results, 1, binSize ** (i+1), (binSize ** (i+2)))
        
        percents = np.array(tourneysWon) / num_wins

        plt.plot(num_wins, percents, marker = "o", label = f"{p.name}")

    plt.xticks(num_wins)
    plt.xscale("log")
    plt.xlabel("Number of Tournament Simulations")
    plt.ylabel("Proportion of Tournaments Won")
    plt.ylim(0, 1)
    plt.title("Convergence of Tournament Win Rates")
    plt.legend()
    plt.show()

def analyzeScoresBarAll(players):
    participants = players.copy()
    participants.sort(key = lambda player: player.init_rating, reverse=True)
    n = len(participants)
    
    for p in participants:
        vals = np.array(p.tournamentScores)
        mean = vals.mean()
        stdev = vals.std()

        frequency = Counter(p.tournamentScores)
        scores = (np.arange(1, 2 * n))/2
        ticks = np.arange(n)
        results = []
        for i in range(2 * n - 1):
            results.append(frequency[i/2])
        percents = np.array(results)
        percents = (percents / percents.sum())

        colorMap = plt.colormaps['Blues']
        norm = plt.Normalize(min(scores), max(scores))
        barColors = colorMap(norm(scores))
        plt.xticks(ticks)
        plt.bar(scores, percents, color = barColors, width = 0.5)
        plt.title(f"{p.name} Tournament Scores ({round(p.init_rating)})")
        plt.text(0.95, 0.95, rf"$\mu = {mean:.2f}$" "\n" rf"$\sigma = {stdev:.2f}$", ha="right", va="top",transform=plt.gca().transAxes)
        plt.show()


In [ ]:
P6.tournament_expected_score(playersList)
P6.analyzeScoresBar()

In [ ]:
tournament_expected_score_rtg(P6.init_rating, P6.index)

In [ ]:
analyzePerformancesLineAll(playersList)

In [ ]:
analyzePerformancesLineAll(playersList)
# analyzePerformancesBarAll(playersList)
for p in playersList:
    p.analyzePerformanceLine()
    p.analyzeRankingsBar()

In [ ]:
#todo 
# - fix the RR pairing system, gives 35 free elo to players who play white a lot
# - implement swiss pairing system
# - add tpr binary search calculation, and subsequent analysis 
# - add multibar chart analysis
# - expected vs actual performance distribution analysis
# - track new ratings after tournament, create graphs

In [ ]:
# import cProfile

# cProfile.run("simulateTournaments(10000, 'RR', playersList)")